<a href="https://colab.research.google.com/github/Rukshana-S/hiver-spotify-ai-support-agent/blob/main/01_spotify_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Hiver Spotify AI Support Agent

## Phase 1: Dataset Loading

In [8]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/Hiver_Spotify_AI/data/twcs.csv"

df = pd.read_csv(DATA_PATH)

In [9]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 7 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   tweet_id                 1048575 non-null  int64  
 1   author_id                1048575 non-null  object 
 2   inbound                  1048575 non-null  bool   
 3   created_at               1048575 non-null  object 
 4   text                     1048575 non-null  object 
 5   response_tweet_id        683860 non-null   object 
 6   in_response_to_tweet_id  767999 non-null   float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 49.0+ MB


In [11]:
df.isnull().sum()

,0
tweet_id,0
author_id,0
inbound,0
created_at,0
text,0
response_tweet_id,364715
in_response_to_tweet_id,280576


In [12]:
df.to_csv(
    "/content/drive/MyDrive/Hiver_Spotify_AI/processed/raw_loaded.csv",
    index=False
)

In [13]:
import pandas as pd
import numpy as np
df = pd.read_csv(DATA_PATH)

In [14]:
print(df.shape)

(1048575, 7)


In [15]:
print(df["author_id"].nunique())

273748


In [16]:
company_accounts = (
    df[df["inbound"] == False]["author_id"]
    .value_counts()
    .head(30)
)

company_accounts

,count
author_id,
AmazonHelp,79139
AppleSupport,32945
Uber_Support,21411
Delta,16372
SpotifyCares,14328
AmericanAir,14324
Tesco,11430
British_Airways,11261
comcastcares,10860


In [17]:
spotify_accounts = df[
    df["author_id"]
      .astype(str)
      .str.contains("spotify", case=False, na=False)
]["author_id"].value_counts()

spotify_accounts

,count
author_id,
SpotifyCares,14328


In [18]:
summary = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Unique Authors"
    ],
    "Value": [
        len(df),
        len(df.columns),
        df["author_id"].nunique()
    ]
})

summary

,Metric,Value
0,Total Rows,1048575
1,Total Columns,7
2,Unique Authors,273748


In [19]:
summary.to_csv(
    "/content/drive/MyDrive/Hiver_Spotify_AI/processed/dataset_summary.csv",
    index=False
)

In [20]:
spotify_accounts

,count
author_id,
SpotifyCares,14328


## Phase 2: Spotify Conversation Extraction

In [21]:
spotify_company = df[df["author_id"] == "SpotifyCares"].copy()

print("Spotify company tweets:", len(spotify_company))
spotify_company.head()

Spotify company tweets: 14328


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
540,848,SpotifyCares,False,Tue Oct 31 22:28:16 +0000 2017,@115887 Hmm. Can you try restarting your devic...,849,850.0
542,851,SpotifyCares,False,Tue Oct 31 23:39:03 +0000 2017,@115887 Could you send us a DM with your accou...,NaN,849.0
544,852,SpotifyCares,False,Tue Oct 31 21:04:13 +0000 2017,"@115887 Thanks. Just to be sure, are you Free ...",850,853.0
546,854,SpotifyCares,False,Tue Oct 31 19:36:16 +0000 2017,"@115887 Hey! What device, operating system, an...",853,855.0
548,856,SpotifyCares,False,Tue Oct 31 22:25:16 +0000 2017,@115889 Got it. It's not possible at the momen...,857,858.0


In [22]:
customer_tweet_ids = (
    spotify_company["in_response_to_tweet_id"]
    .dropna()
    .astype(int)
    .unique()
)

print("Customer tweets connected to Spotify:", len(customer_tweet_ids))

Customer tweets connected to Spotify: 13969


In [23]:
spotify_customers = df[df["tweet_id"].isin(customer_tweet_ids)].copy()

print("Connected customer tweets:", len(spotify_customers))
spotify_customers.head()

Connected customer tweets: 13952


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
541,849,115887,True,Tue Oct 31 23:36:20 +0000 2017,@SpotifyCares doesn’t work and i even tried de...,851,848.0
543,850,115887,True,Tue Oct 31 21:41:37 +0000 2017,@SpotifyCares Premium &amp; when i️ have it on...,848,852.0
545,853,115887,True,Tue Oct 31 19:57:59 +0000 2017,@SpotifyCares iphone 7+ and i have the most re...,852,854.0
547,855,115887,True,Tue Oct 31 19:10:51 +0000 2017,i’m pissed my @115888 shuffle and repeat butto...,854,NaN
549,857,115889,True,Wed Nov 01 07:40:06 +0000 2017,"@SpotifyCares Yes, multiple times. No changes....",859,856.0


In [24]:
spotify_twitter = pd.concat(
    [spotify_company, spotify_customers],
    ignore_index=True
)

spotify_twitter = spotify_twitter.drop_duplicates(subset="tweet_id")

print("Total Spotify conversation tweets:", len(spotify_twitter))

Total Spotify conversation tweets: 28250


In [25]:
spotify_twitter = spotify_twitter.sort_values("created_at")

spotify_twitter.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
23618,787488,307849,True,Fri Apr 07 10:36:18 +0000 2017,@115888 (3) their playlists bc ur algorithm ca...,787487,NaN
9498,787487,SpotifyCares,False,Fri Apr 07 11:25:53 +0000 2017,@307849 Hey Will! We’ve made some improvements...,787485,787488.0
23617,787485,307849,True,Fri Apr 07 12:09:32 +0000 2017,@SpotifyCares Improvements haven't been notice...,787482,787487.0
9496,787482,SpotifyCares,False,Fri Apr 07 14:58:17 +0000 2017,@307849 We're sorry you feel that way. We appr...,"7,87,48,37,87,484",787485.0
17544,301485,187918,True,Fri Aug 11 17:05:40 +0000 2017,@SpotifyCares before I already asked about if ...,301484,NaN


In [26]:
OUTPUT_PATH = "/content/drive/MyDrive/Hiver_Spotify_AI/processed/spotify_twitter.csv"

spotify_twitter.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

Saved to: /content/drive/MyDrive/Hiver_Spotify_AI/processed/spotify_twitter.csv


In [27]:
spotify_twitter["response_tweet_id"].dropna().head(10)

,response_tweet_id
23618,787487
9498,787485
23617,787482
9496,"7,87,48,37,87,484"
17544,301484
3306,301483
17543,301482
3305,301481
17542,301480
3304,301478


In [29]:
def split_responses(value):
    if pd.isna(value):
        return []
    s = str(value)
    # Remove any character that is not a digit or a comma
    cleaned_s = "".join(filter(lambda x: x.isdigit() or x == ',', s))
    if not cleaned_s: # If after cleaning, the string is empty or only commas
        return []

    response_ids = []
    for part in cleaned_s.split(","):
        part = part.strip()
        if part.isdigit(): # Check if the part consists only of digits
            response_ids.append(int(part))
    return response_ids

spotify_twitter["response_list"] = spotify_twitter["response_tweet_id"].apply(split_responses)

spotify_twitter[["tweet_id", "response_tweet_id", "response_list"]].head()

,tweet_id,response_tweet_id,response_list
23618,787488,787487,[787487]
9498,787487,787485,[787485]
23617,787485,787482,[787482]
9496,787482,"7,87,48,37,87,484","[7, 87, 48, 37, 87, 484]"
17544,301485,301484,[301484]


In [30]:
tweet_lookup = spotify_twitter.set_index("tweet_id").to_dict("index")

len(tweet_lookup)

28250

In [31]:
def build_thread(start_id):
    thread = []
    current = start_id
    visited = set()

    while current in tweet_lookup and current not in visited:
        visited.add(current)

        row = tweet_lookup[current]

        thread.append({
            "tweet_id": current,
            "author": row["author_id"],
            "inbound": row["inbound"],
            "text": row["text"]
        })

        responses = row["response_list"]

        if responses:
            current = responses[0]      # follow first reply for now
        else:
            break

    return thread

In [32]:
conversation_roots = spotify_twitter[
    (spotify_twitter["inbound"] == True) &
    (spotify_twitter["in_response_to_tweet_id"].isna())
]["tweet_id"].tolist()

print("Conversation roots:", len(conversation_roots))

Conversation roots: 7834


In [33]:
threads = []

for root in conversation_roots:
    thread = build_thread(root)

    if len(thread) >= 2:     # customer + Spotify
        threads.append(thread)

print("Complete conversations:", len(threads))

Complete conversations: 7336


In [34]:
conversation_pairs = []

for thread in threads:
    for i in range(len(thread)-1):
        if thread[i]["inbound"] and not thread[i+1]["inbound"]:
            conversation_pairs.append({
                "customer_message": thread[i]["text"],
                "spotify_reply": thread[i+1]["text"]
            })

conversation_pairs = pd.DataFrame(conversation_pairs)

conversation_pairs.head()

,customer_message,spotify_reply
0,@115888 (3) their playlists bc ur algorithm ca...,@307849 Hey Will! We’ve made some improvements...
1,@SpotifyCares Improvements haven't been notice...,@307849 We're sorry you feel that way. We appr...
2,@SpotifyCares before I already asked about if ...,"@187918 Hi there! At this moment, it's not pos..."
3,@SpotifyCares There are so many issues that th...,@187918 That's not cool! What specifically is ...
4,@SpotifyCares For example in playlists I no lo...,@187918 Which device and version of Spotify ar...


In [35]:
conversation_pairs.to_csv(
    "/content/drive/MyDrive/Hiver_Spotify_AI/processed/conversation_pairs.csv",
    index=False
)

In [36]:
# Create a graph: tweet_id -> list of reply tweet IDs

reply_graph = {}

for _, row in spotify_twitter.iterrows():
    tweet_id = row["tweet_id"]
    replies = row["response_list"]

    reply_graph[tweet_id] = replies

print("Tweets in graph:", len(reply_graph))

Tweets in graph: 28250


In [37]:
# Tweets that have multiple replies

branching = spotify_twitter[
    spotify_twitter["response_list"].apply(len) > 1
]

print("Branching conversations:", len(branching))

branching[["tweet_id","response_tweet_id"]].head()

Branching conversations: 1298


,tweet_id,response_tweet_id
9496,787482,"7,87,48,37,87,484"
19283,459150,"4,59,14,94,59,151"
5073,458061,"4,58,06,24,58,063"
1062,106311,"1,06,31,21,06,313"
19291,461455,"4,61,45,64,61,45,74,61,00,00,00,00,00,00,00,00..."


In [38]:
def build_tree(start_id):
    stack=[start_id]
    visited=set()
    conversation=[]

    while stack:
        current=stack.pop()

        if current in visited:
            continue

        visited.add(current)

        if current not in tweet_lookup:
            continue

        row=tweet_lookup[current]

        conversation.append({
            "tweet_id":current,
            "author":row["author_id"],
            "inbound":row["inbound"],
            "text":row["text"]
        })

        for reply in reversed(reply_graph.get(current,[])):
            stack.append(reply)

    return conversation

In [39]:
roots=spotify_twitter[
    (spotify_twitter["inbound"]==True) &
    (spotify_twitter["in_response_to_tweet_id"].isna())
]["tweet_id"].tolist()

print("Root conversations:",len(roots))

Root conversations: 7834


In [40]:
all_threads=[]

for root in roots:
    thread=build_tree(root)

    if len(thread)>=2:
        all_threads.append(thread)

print("Complete conversation trees:",len(all_threads))

Complete conversation trees: 7350


In [44]:
pairs = []

for _, row in spotify_twitter.iterrows():

    parent_id = row["in_response_to_tweet_id"]

    if pd.isna(parent_id):
        continue

    parent_id = int(parent_id)

    if parent_id not in tweet_lookup:
        continue

    parent = tweet_lookup[parent_id]
    child = row

    # Customer -> Spotify reply
    if parent["inbound"] and not child["inbound"]:
        pairs.append({
            "customer_message": parent["text"],
            "spotify_reply": child["text"]
        })

conversation_pairs = pd.DataFrame(pairs)

print(conversation_pairs.shape)
conversation_pairs.head()

(14269, 2)


,customer_message,spotify_reply
0,@115888 (3) their playlists bc ur algorithm ca...,@307849 Hey Will! We’ve made some improvements...
1,@SpotifyCares Improvements haven't been notice...,@307849 We're sorry you feel that way. We appr...
2,@SpotifyCares before I already asked about if ...,"@187918 Hi there! At this moment, it's not pos..."
3,@SpotifyCares There are so many issues that th...,@187918 That's not cool! What specifically is ...
4,@SpotifyCares For example in playlists I no lo...,@187918 Which device and version of Spotify ar...


In [42]:
conversation_pairs.to_csv(
    "/content/drive/MyDrive/Hiver_Spotify_AI/processed/conversation_pairs.csv",
    index=False
)

print("conversation_pairs.csv saved")

conversation_pairs.csv saved


In [43]:
# Create a lookup for every tweet
tweet_lookup = spotify_twitter.set_index("tweet_id").to_dict("index")


In [45]:
conversation_pairs.to_csv(
    "/content/drive/MyDrive/Hiver_Spotify_AI/processed/conversation_pairs.csv",
    index=False
)

print("Updated conversation_pairs.csv saved.")

Updated conversation_pairs.csv saved.
